In [1]:
# %% [Cell 1]
import os
import cv2
import numpy as np
import mediapipe as mp
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelEncoder
import pickle

# %% [Cell 2]
# Paths (update these to your actual folders)
TRAIN_PATH = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET\TRAIN"
TEST_PATH = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET\TEST"

# %% [Cell 3]
# Initialize MediaPipe Pose
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)

# %% [Cell 4]
def extract_keypoints(image_path):
    image = cv2.imread(image_path)
    if image is None:
        print(f"Warning: Unable to read {image_path}")
        return None
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)
    if not results.pose_landmarks:
        return None  # No pose detected
    keypoints = []
    for lm in results.pose_landmarks.landmark:
        keypoints.extend([lm.x, lm.y, lm.z, lm.visibility])
    return keypoints  # 33 landmarks * 4 = 132 features

# %% [Cell 5]
def load_data(path):
    X = []
    y = []
    labels = sorted(os.listdir(path))
    print(f"Found classes: {labels}")
    for label in labels:
        class_folder = os.path.join(path, label)
        if not os.path.isdir(class_folder):
            continue
        for img_file in os.listdir(class_folder):
            img_path = os.path.join(class_folder, img_file)
            keypoints = extract_keypoints(img_path)
            if keypoints is not None:
                X.append(keypoints)
                y.append(label)
    return np.array(X), np.array(y), labels

# %% [Cell 6]
# Load train and test data
print("Extracting train data keypoints...")
X_train, y_train, class_names = load_data(TRAIN_PATH)

print("Extracting test data keypoints...")
X_test, y_test, _ = load_data(TEST_PATH)

print(f"Train samples: {len(X_train)}, Test samples: {len(X_test)}")

# %% [Cell 7]
# Encode labels
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc = le.transform(y_test)

# %% [Cell 8]
# Train classifier
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_train, y_train_enc)

# %% [Cell 9]
# Test accuracy
test_preds = clf.predict(X_test)
print("Test Accuracy:", accuracy_score(y_test_enc, test_preds))

# %% [Cell 10]
# Save model and label encoder
with open('yoga_pose_classifier.pkl', 'wb') as f:
    pickle.dump(clf, f)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

print("Model and label encoder saved.")

Extracting train data keypoints...
Found classes: ['downdog', 'goddess', 'plank', 'tree', 'warrior2']
Extracting test data keypoints...
Found classes: ['downdog', 'goddess', 'plank', 'tree', 'warrior2']
Train samples: 1042, Test samples: 465
Test Accuracy: 0.9827956989247312
Model and label encoder saved.


In [2]:
import joblib
# Save training data for reuse
joblib.dump(X_train, 'X_train.pkl')
joblib.dump(X_test, 'X_test.pkl')
joblib.dump(y_test, 'y_train.pkl')
joblib.dump(y_test, 'y_test.pkl')

['y_test.pkl']

In [3]:
# 📦 Install dependencies if not done
# pip install mediapipe scikit-learn matplotlib seaborn opencv-python skl2onnx joblib

import os
import cv2
import numpy as np
import joblib
import mediapipe as mp
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

# === Load Classifier & Encoder ===
clf = joblib.load('yoga_pose_classifier.pkl')          # Random Forest
le = joblib.load('label_encoder.pkl')                  # LabelEncoder
X_train = joblib.load('X_train.pkl')                   # Save these during training
X_test = joblib.load('X_test.pkl')
y_test = joblib.load('y_test.pkl')

# === MediaPipe Setup ===
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True)
mp_drawing = mp.solutions.drawing_utils

# === STEP 1: Single Image Visualization ===
def plot_prediction(image_path):
    img = cv2.imread(image_path)
    if img is None:
        print("Invalid image path.")
        return
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = pose.process(img_rgb)

    if results.pose_landmarks:
        landmarks = [coord for lm in results.pose_landmarks.landmark for coord in (lm.x, lm.y, lm.z)]
        pred = clf.predict([landmarks])[0]
        label = le.inverse_transform([pred])[0]
        mp_drawing.draw_landmarks(img_rgb, results.pose_landmarks, mp_pose.POSE_CONNECTIONS)
        plt.imshow(img_rgb)
        plt.title(f'Prediction: {label}', fontsize=14)
        plt.axis('off')
        plt.show()
    else:
        print("No pose detected.")

# Example usage
# plot_prediction('/content/drive/MyDrive/test/Tree_Pose_or_Vrksasana_/Tree_Pose_or_Vrksasana__image_123.jpg')


# === STEP 2: Confusion Matrix ===
def show_confusion_matrix():
    y_pred = clf.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=le.classes_)
    fig, ax = plt.subplots(figsize=(16, 16))
    disp.plot(ax=ax, xticks_rotation='vertical')
    plt.title("Confusion Matrix", fontsize=18)
    plt.grid(False)
    plt.show()

# Example usage
# show_confusion_matrix()


# === STEP 3: Convert Model to ONNX ===
def convert_to_onnx(model, X_sample):
    initial_type = [('float_input', FloatTensorType([None, len(X_sample[0])]))]
    onnx_model = convert_sklearn(model, initial_types=initial_type)
    with open("yoga_pose_classifier.onnx", "wb") as f:
        f.write(onnx_model.SerializeToString())
    print("✅ Model converted and saved as yoga_pose_classifier.onnx")

# Example usage
# convert_to_onnx(clf, X_train)


# === STEP 4: Real-Time Pose Classification ===
def run_realtime_pose():
    cap = cv2.VideoCapture(0)
    realtime_pose = mp_pose.Pose()
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        img_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = realtime_pose.process(img_rgb)

        if result.pose_landmarks:
            keypoints = [coord for lm in result.pose_landmarks.landmark for coord in (lm.x, lm.y, lm.z)]
            if len(keypoints) == len(X_train[0]):
                pred = clf.predict([keypoints])[0]
                label = le.inverse_transform([pred])[0]
                cv2.putText(frame, f'{label}', (30, 60), cv2.FONT_HERSHEY_SIMPLEX, 1.5, (0, 255, 0), 3)

            mp_drawing.draw_landmarks(frame, result.pose_landmarks, mp_pose.POSE_CONNECTIONS)

        cv2.imshow('Yoga Pose Detector', frame)
        if cv2.waitKey(10) & 0xFF == ord('q'):
            break

    cap.release()
    cv2.destroyAllWindows()

# Example usage
# run_realtime_pose()


In [4]:
import cv2
import numpy as np
import joblib
import mediapipe as mp
import os

# === Load trained model and label encoder ===
clf = joblib.load('yoga_pose_classifier.pkl')
le = joblib.load('label_encoder.pkl')

# === Function to extract keypoints from a single image ===
def extract_keypoints_from_image(image_path):
    mp_pose = mp.solutions.pose
    pose = mp_pose.Pose(static_image_mode=True)

    img = cv2.imread(image_path)
    if img is None:
        raise ValueError(f"Image not found at {image_path}")

    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    results = pose.process(img_rgb)

    if not results.pose_landmarks:
        raise ValueError("No pose landmarks detected in the image")

    keypoints = []
    for lm in results.pose_landmarks.landmark:
        keypoints.extend([lm.x, lm.y, lm.z, lm.visibility])

    keypoints = np.array(keypoints).reshape(1, -1)
    pose.close()
    return keypoints

# === Step 1: Provide local file path ===
# Replace this with your image path
image_path = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET\TRAIN\goddess\00000096.jpg"

if not os.path.exists(image_path):
    raise FileNotFoundError(f"No image found at {image_path}")

print(f"Processing file: {image_path}")

# === Step 2: Extract keypoints/features ===
X_single = extract_keypoints_from_image(image_path)

# === Step 3: Predict pose class ===
y_pred_single = clf.predict(X_single)
pose_predicted = le.inverse_transform(y_pred_single)

print(f"Predicted Yoga Pose: {pose_predicted[0]}")

Processing file: C:\Users\dell\Desktop\Yoga pose detection\DATASET\TRAIN\goddess\00000096.jpg
Predicted Yoga Pose: goddess


In [5]:
import cv2
import mediapipe as mp
import numpy as np
import os

def draw_pose_landmarks(image_array):
    mp_pose = mp.solutions.pose
    mp_drawing = mp.solutions.drawing_utils

    with mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5) as pose:
        results = pose.process(cv2.cvtColor(image_array, cv2.COLOR_BGR2RGB))
        if not results.pose_landmarks:
            print("No pose landmarks detected.")
            return None

        mp_drawing.draw_landmarks(
            image_array, results.pose_landmarks, mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=3),
            mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2)
        )
        return image_array

# === Step 1: Provide local file path ===
image_path = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET\TRAIN\goddess\00000096.jpg"

if not os.path.exists(image_path):
    raise FileNotFoundError(f"No image found at {image_path}")

# === Step 2: Read image ===
img = cv2.imread(image_path)

# === Step 3: Draw pose landmarks ===
img_with_landmarks = draw_pose_landmarks(img)

if img_with_landmarks is not None:
    # === Step 4: Save output image ===
    output_path = "output_pose.png"
    cv2.imwrite(output_path, img_with_landmarks)
    print(f"Pose landmarks drawn and saved as {output_path}")

    # === Step 5: Show image in a window ===
    cv2.imshow("Pose Landmarks", img_with_landmarks)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

Pose landmarks drawn and saved as output_pose.png


In [6]:
import cv2
import mediapipe as mp
import numpy as np
import pickle
import os

# === Load trained model and label encoder ===
with open('yoga_pose_classifier.pkl', 'rb') as f:
    clf = pickle.load(f)
with open('label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# === Functions ===
def extract_keypoints(image_array):
    with mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5) as pose:
        image_rgb = cv2.cvtColor(image_array, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)
        if not results.pose_landmarks:
            return None, None
        keypoints = []
        for lm in results.pose_landmarks.landmark:
            keypoints.extend([lm.x, lm.y, lm.z, lm.visibility])
        return np.array(keypoints).reshape(1, -1), results.pose_landmarks

def draw_pose_landmarks(image_array, landmarks):
    mp_drawing.draw_landmarks(
        image_array, landmarks, mp_pose.POSE_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(0,255,0), thickness=2, circle_radius=3),
        mp_drawing.DrawingSpec(color=(0,0,255), thickness=2))

# === Step 1: Provide local image path ===
image_path = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET\TRAIN\goddess\00000096.jpg"

if not os.path.exists(image_path):
    raise FileNotFoundError(f"No image found at {image_path}")

img = cv2.imread(image_path)

# === Step 2: Extract keypoints & landmarks ===
keypoints, landmarks = extract_keypoints(img)

if keypoints is None:
    print("No pose detected in the image.")
    cv2.imwrite("output_pose.png", img)
    cv2.imshow("Pose Detection", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()
else:
    # === Step 3: Predict the pose label ===
    pred_label_idx = clf.predict(keypoints)[0]
    pred_label = le.inverse_transform([pred_label_idx])[0]

    # === Step 4: Draw landmarks ===
    draw_pose_landmarks(img, landmarks)

    # === Step 5: Put text label on the image ===
    cv2.putText(img, f"Pose: {pred_label}", (30, 50), cv2.FONT_HERSHEY_SIMPLEX,
                1.2, (0, 255, 0), 3, cv2.LINE_AA)

    # === Step 6: Save & show image ===
    output_path = "output_pose.png"
    cv2.imwrite(output_path, img)

    print(f"Detected Pose: {pred_label}")
    cv2.imshow("Pose Detection", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

Detected Pose: goddess


In [7]:
import os
import cv2
import numpy as np
import mediapipe as mp
import pickle
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder

# ---------------- PATHS ----------------
NEW_DATA_PATH = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET\TEST"  # New images folder
OLD_X_PATH = "X_train.pkl"      # Previously saved features
OLD_Y_PATH = "y_train.pkl"      # Previously saved labels
MODEL_PATH = "yoga_pose_classifier_updated.pkl"
LABEL_ENCODER_PATH = "label_encoder_updated.pkl"

# ---------------- Initialize MediaPipe Pose ----------------
mp_pose = mp.solutions.pose
pose = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)

# ---------------- Function to Extract Keypoints ----------------
def extract_keypoints(image_path):
    image = cv2.imread(image_path)
    if image is None:
        return None
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose.process(image_rgb)
    
    keypoints = []
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            keypoints.extend([lm.x, lm.y, lm.z, lm.visibility])
    
    # Pad missing landmarks with zeros to ensure length 132
    if len(keypoints) < 132:
        keypoints.extend([0.0] * (132 - len(keypoints)))
    
    return keypoints

# ---------------- Function to Load New Data ----------------
def load_new_data(path):
    X_new = []
    y_new = []

    for label in sorted(os.listdir(path)):
        class_path = os.path.join(path, label)
        if not os.path.isdir(class_path):
            continue

        img_files = [f for f in os.listdir(class_path) if f.lower().endswith(('.jpg','.jpeg','.png'))]

        for img_file in img_files:
            img_path = os.path.join(class_path, img_file)
            keypoints = extract_keypoints(img_path)
            if keypoints is not None:
                X_new.append(keypoints)
                y_new.append(label)

    X_new = np.array(X_new, dtype=np.float32)
    y_new = np.array(y_new)

    # Safety check: truncate if mismatch occurs
    if len(X_new) != len(y_new):
        print(f"Warning: Mismatch detected in new data! Truncating to smaller length")
        min_len = min(len(X_new), len(y_new))
        X_new = X_new[:min_len]
        y_new = y_new[:min_len]

    return X_new, y_new

# ---------------- Load Old Data ----------------
print("Loading old training data...")
X_old = joblib.load(OLD_X_PATH)
y_old = joblib.load(OLD_Y_PATH)

# Trim old features if labels are fewer
if X_old.shape[0] != y_old.shape[0]:
    print(f"Warning: X_old has {X_old.shape[0]} samples but y_old has {y_old.shape[0]} labels. Trimming X_old to match y_old.")
    X_old = X_old[:y_old.shape[0]]

print(f"Old data: X_old={X_old.shape}, y_old={y_old.shape}")

# ---------------- Load New Data ----------------
print("Extracting keypoints from new data...")
X_new, y_new = load_new_data(NEW_DATA_PATH)
print(f"New data: X_new={X_new.shape}, y_new={y_new.shape}")

# ---------------- Combine Old and New ----------------
if X_new.shape[0] == 0:
    print("No valid new samples found. Using only old data.")
    X_combined = X_old
    y_combined = y_old
else:
    X_combined = np.concatenate([X_old, X_new], axis=0)
    y_combined = np.concatenate([y_old, y_new], axis=0)

print(f"Combined shapes: X_combined={X_combined.shape}, y_combined={y_combined.shape}")

# Final safety check
if X_combined.shape[0] != y_combined.shape[0]:
    min_len = min(X_combined.shape[0], y_combined.shape[0])
    print(f"Warning: Final mismatch! Truncating combined data to {min_len} samples.")
    X_combined = X_combined[:min_len]
    y_combined = y_combined[:min_len]

# ---------------- Encode Labels ----------------
le = LabelEncoder()
y_encoded = le.fit_transform(y_combined)

# ---------------- Train RandomForestClassifier ----------------
print("Training RandomForestClassifier on combined data...")
clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X_combined, y_encoded)

# ---------------- Save Updated Model and Label Encoder ----------------
with open(MODEL_PATH, 'wb') as f:
    pickle.dump(clf, f)
with open(LABEL_ENCODER_PATH, 'wb') as f:
    pickle.dump(le, f)

print("Training complete. Model and label encoder saved successfully.")

Loading old training data...
Old data: X_old=(465, 132), y_old=(465,)
Extracting keypoints from new data...
New data: X_new=(470, 132), y_new=(470,)
Combined shapes: X_combined=(935, 132), y_combined=(935,)
Training RandomForestClassifier on combined data...
Training complete. Model and label encoder saved successfully.


In [8]:
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder

# Encode labels
label_encoder = LabelEncoder()
y_train = label_encoder.fit_transform(y_train)

# Build model
model = tf.keras.Sequential([
    tf.keras.layers.Dense(128, activation='relu', input_shape=(132,)),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dense(len(set(y_train)), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

# Train model
model.fit(X_train, y_train, epochs=10)

# Save model
model.save("yoga_pose_classifier.h5")

Epoch 1/10
33/33 [==============================] - 2s 4ms/step - loss: 1.3677 - accuracy: 0.4597
Epoch 2/10
33/33 [==============================] - 0s 4ms/step - loss: 1.0219 - accuracy: 0.6267
Epoch 3/10
33/33 [==============================] - 0s 4ms/step - loss: 0.8282 - accuracy: 0.7111
Epoch 4/10
33/33 [==============================] - 0s 4ms/step - loss: 0.7489 - accuracy: 0.7226
Epoch 5/10
33/33 [==============================] - 0s 3ms/step - loss: 0.6565 - accuracy: 0.7764
Epoch 6/10
33/33 [==============================] - 0s 6ms/step - loss: 0.5746 - accuracy: 0.8196
Epoch 7/10
33/33 [==============================] - 0s 4ms/step - loss: 0.5317 - accuracy: 0.8349
Epoch 8/10
33/33 [==============================] - 0s 4ms/step - loss: 0.4786 - accuracy: 0.8589
Epoch 9/10
33/33 [==============================] - 0s 4ms/step - loss: 0.4425 - accuracy: 0.8618
Epoch 10/10
33/33 [==============================] - 0s 3ms/step - loss: 0.4031 - accuracy: 0.8810


c:\Users\dell\Desktop\Yoga pose detection\torch_env\lib\site-packages\keras\src\engine\training.py:3000: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [9]:
!pip install mediapipe

  Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl.metadata (60 kB)
Using cached numpy-2.2.6-cp310-cp310-win_amd64.whl (12.9 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 1.26.4
    Uninstalling numpy-1.26.4:
      Successfully uninstalled numpy-1.26.4


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
onnx 1.20.1 requires protobuf>=4.25.1, but you have protobuf 3.20.3 which is incompatible.
tensorflow-intel 2.13.0 requires numpy<=1.24.3,>=1.22, but you have numpy 2.2.6 which is incompatible.
tensorflow-intel 2.13.0 requires typing-extensions<4.6.0,>=3.6.6, but you have typing-extensions 4.12.2 which is incompatible.


In [10]:
import os
import cv2
import numpy as np
import mediapipe as mp
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader
import pickle

# ==============================
# CONFIGURATION
# ==============================

TRAIN_PATH = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET\TRAIN"
TEST_PATH = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET\TEST"
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 32
EPOCHS = 25

# ==============================
# MEDIAPIPE POSE
# ==============================

mp_pose = mp.solutions.pose
pose_detector = mp_pose.Pose(
    static_image_mode=True,
    min_detection_confidence=0.5
)

# ==============================
# KEYPOINT EXTRACTION
# ==============================

def extract_keypoints(image_path):

    image = cv2.imread(image_path)

    if image is None:
        print(f"Warning: Unable to read {image_path}")
        return None

    image = cv2.resize(image, IMAGE_SIZE)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    results = pose_detector.process(image_rgb)

    if not results.pose_landmarks:
        return None

    keypoints = []

    for lm in results.pose_landmarks.landmark:
        keypoints.extend([lm.x, lm.y, lm.z, lm.visibility])

    return keypoints  # 33 * 4 = 132 features


# ==============================
# LOAD DATASET
# ==============================

def load_data(path):

    X = []
    y = []

    labels = sorted(os.listdir(path))
    print("Found classes:", labels)

    for label in labels:

        class_folder = os.path.join(path, label)

        if not os.path.isdir(class_folder):
            continue

        for file in os.listdir(class_folder):

            img_path = os.path.join(class_folder, file)

            keypoints = extract_keypoints(img_path)

            if keypoints is not None:
                X.append(keypoints)
                y.append(label)

    return np.array(X, dtype=np.float32), np.array(y), labels


# ==============================
# LOAD TRAIN + TEST DATA
# ==============================

print("Loading training data...")
X_train, y_train, class_names = load_data(TRAIN_PATH)

print("Loading test data...")
X_test, y_test, _ = load_data(TEST_PATH)


# ==============================
# LABEL ENCODING
# ==============================

label_encoder = LabelEncoder()

y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)

num_classes = len(label_encoder.classes_)

# Save label encoder
with open("label_encoder.pkl", "wb") as f:
    pickle.dump(label_encoder, f)


# ==============================
# PYTORCH DATASET
# ==============================

class PoseDataset(Dataset):

    def __init__(self, X, y):

        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


train_loader = DataLoader(
    PoseDataset(X_train, y_train_enc),
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    PoseDataset(X_test, y_test_enc),
    batch_size=BATCH_SIZE
)


# ==============================
# MODEL DEFINITION
# ==============================

class PoseClassifier(nn.Module):

    def __init__(self, input_dim=132, hidden1=256, hidden2=128, output_dim=num_classes):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(hidden2, output_dim)
        )

    def forward(self, x):
        return self.net(x)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PoseClassifier().to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

loss_fn = nn.CrossEntropyLoss()


# ==============================
# TRAINING
# ==============================

print("Training model...")

for epoch in range(EPOCHS):

    model.train()
    running_loss = 0

    for X_batch, y_batch in train_loader:

        X_batch = X_batch.to(device)
        y_batch = y_batch.to(device)

        optimizer.zero_grad()

        predictions = model(X_batch)

        loss = loss_fn(predictions, y_batch)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)

    print(f"Epoch {epoch+1}/{EPOCHS}  Loss: {avg_loss:.4f}")


# ==============================
# TEST ACCURACY
# ==============================

model.eval()

y_preds = []
y_true = []

with torch.no_grad():

    for X_batch, y_batch in test_loader:

        X_batch = X_batch.to(device)

        predictions = model(X_batch)

        preds = torch.argmax(predictions, dim=1)

        y_preds.extend(preds.cpu().numpy())
        y_true.extend(y_batch.numpy())

test_accuracy = accuracy_score(y_true, y_preds)

print("Test Accuracy:", test_accuracy)


# ==============================
# SAVE MODEL
# ==============================

torch.save(model.state_dict(), "pose_classifier.pth")

print("Model saved: pose_classifier.pth")


# ==============================
# EXPORT TORCHSCRIPT
# ==============================

example_input = torch.rand(1, 132).to(device)

scripted_model = torch.jit.trace(model, example_input)

scripted_model.save("pose_classifier_scripted.pt")

print("TorchScript model saved: pose_classifier_scripted.pt")

Loading training data...
Found classes: ['downdog', 'goddess', 'plank', 'tree', 'warrior2']
Loading test data...
Found classes: ['downdog', 'goddess', 'plank', 'tree', 'warrior2']
Training model...
Epoch 1/25  Loss: 1.4616
Epoch 2/25  Loss: 1.0501
Epoch 3/25  Loss: 0.8872
Epoch 4/25  Loss: 0.8216
Epoch 5/25  Loss: 0.7666
Epoch 6/25  Loss: 0.7071
Epoch 7/25  Loss: 0.6194
Epoch 8/25  Loss: 0.5801
Epoch 9/25  Loss: 0.5525
Epoch 10/25  Loss: 0.5075
Epoch 11/25  Loss: 0.4708
Epoch 12/25  Loss: 0.4680
Epoch 13/25  Loss: 0.4383
Epoch 14/25  Loss: 0.4002
Epoch 15/25  Loss: 0.3941
Epoch 16/25  Loss: 0.3732
Epoch 17/25  Loss: 0.3857
Epoch 18/25  Loss: 0.3694
Epoch 19/25  Loss: 0.3374
Epoch 20/25  Loss: 0.3367
Epoch 21/25  Loss: 0.3367
Epoch 22/25  Loss: 0.3280
Epoch 23/25  Loss: 0.3408
Epoch 24/25  Loss: 0.3203
Epoch 25/25  Loss: 0.2911
Test Accuracy: 0.9543478260869566
Model saved: pose_classifier.pth
TorchScript model saved: pose_classifier_scripted.pt


In [12]:
import torch
from torch.utils.mobile_optimizer import optimize_for_mobile
from torchvision import models

def main():
    # Load model (example: VGG19 pretrained)
    model = models.vgg19(pretrained=True)

    # Move model to CPU
    model.cpu()
    model.eval()

    # Create example input on CPU
    example_input = torch.rand(1, 3, 224, 224)

    # Trace the model
    traced_script_module = torch.jit.trace(model, example_input)

    # Optimize for mobile
    optimized_traced_model = optimize_for_mobile(traced_script_module)

    # Save the optimized model
    optimized_traced_model._save_for_lite_interpreter("vgg19.pt")

    print("Model successfully converted and saved as vgg19.pt")

if __name__ == "__main__":
    main()

c:\Users\dell\Desktop\Yoga pose detection\torch_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\dell\Desktop\Yoga pose detection\torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to C:\Users\dell/.cache\torch\hub\checkpoints\vgg19-dcbb9e9d.pth
100.0%


✅ Model successfully converted and saved as vgg19.pt


In [13]:
import os
import cv2
import numpy as np
import mediapipe as mp
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
import pickle

# === CONFIGURATION ===
TRAIN_PATH = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET\TRAIN"   # <-- change this
TEST_PATH  = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET\TEST"    # <-- change this
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 32
EPOCHS = 25

# === MEDIAPIPE ===
mp_pose = mp.solutions.pose
pose_detector = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)

def extract_keypoints(image_path):
    image = cv2.imread(image_path)
    if image is None:
        print(f"Warning: Unable to read {image_path}")
        return None

    image = cv2.resize(image, IMAGE_SIZE)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    results = pose_detector.process(image_rgb)

    if not results.pose_landmarks:
        return None

    keypoints = []
    for lm in results.pose_landmarks.landmark:
        keypoints.extend([lm.x, lm.y, lm.z, lm.visibility])

    return keypoints  # 132 features

def load_data(path):
    X, y = [], []
    labels = sorted(os.listdir(path))
    print(f"Found classes: {labels}")

    for label in labels:
        class_folder = os.path.join(path, label)
        if not os.path.isdir(class_folder):
            continue

        for file in os.listdir(class_folder):
            img_path = os.path.join(class_folder, file)
            keypoints = extract_keypoints(img_path)

            if keypoints is not None:
                X.append(keypoints)
                y.append(label)

    return np.array(X, dtype=np.float32), np.array(y), labels

# === LOAD DATA ===
print("Loading training data...")
X_train, y_train, class_names = load_data(TRAIN_PATH)

print("Loading test data...")
X_test, y_test, _ = load_data(TEST_PATH)

# === SPLIT ===
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train, y_train, test_size=0.2, stratify=y_train, random_state=42
)

# === LABEL ENCODING ===
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_valid_enc = le.transform(y_valid)
y_test_enc  = le.transform(y_test)

num_classes = len(le.classes_)

with open('label_encoder.pkl', 'wb') as f:
    pickle.dump(le, f)

# === DATASET ===
class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(PoseDataset(X_train, y_train_enc), batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(PoseDataset(X_valid, y_valid_enc), batch_size=BATCH_SIZE)
test_loader  = DataLoader(PoseDataset(X_test, y_test_enc), batch_size=BATCH_SIZE)

# === MODEL ===
class PoseClassifier(nn.Module):
    def __init__(self, input_dim=132, hidden1=256, hidden2=128, output_dim=num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, output_dim)
        )

    def forward(self, x):
        return self.net(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PoseClassifier().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# === TRAINING ===
print("Training...")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(Xb)
        loss = loss_fn(preds, yb)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {running_loss/len(train_loader):.4f}")

    # Validation
    model.eval()
    y_preds, y_true = [], []

    with torch.no_grad():
        for Xv, yv in valid_loader:
            Xv, yv = Xv.to(device), yv.to(device)
            preds = model(Xv)

            y_preds.extend(torch.argmax(preds, 1).cpu().numpy())
            y_true.extend(yv.cpu().numpy())

    acc = accuracy_score(y_true, y_preds)
    print(f"Validation Accuracy: {acc:.4f}")

# === TEST ===
model.eval()
y_preds, y_true = [], []

with torch.no_grad():
    for Xt, yt in test_loader:
        Xt, yt = Xt.to(device), yt.to(device)
        preds = model(Xt)

        y_preds.extend(torch.argmax(preds, 1).cpu().numpy())
        y_true.extend(yt.cpu().numpy())

test_acc = accuracy_score(y_true, y_preds)
print(f"Test Accuracy: {test_acc:.4f}")

# === SAVE MODEL ===
torch.save(model.state_dict(), "pose_classifier.pth")
print("Saved: pose_classifier.pth")

# === TORCHSCRIPT EXPORT ===
example_input = torch.rand(1, 132).to(device)
scripted_model = torch.jit.trace(model, example_input)
scripted_model.save("pose_classifier_scripted.pt")

print("Saved: pose_classifier_scripted.pt")

Loading training data...
Found classes: ['downdog', 'goddess', 'plank', 'tree', 'warrior2']
Loading test data...
Found classes: ['downdog', 'goddess', 'plank', 'tree', 'warrior2']
Training...
Epoch 1/25, Loss: 1.4773
Validation Accuracy: 0.5268
Epoch 2/25, Loss: 1.1354
Validation Accuracy: 0.5659
Epoch 3/25, Loss: 0.9801
Validation Accuracy: 0.5902
Epoch 4/25, Loss: 0.8775
Validation Accuracy: 0.7463
Epoch 5/25, Loss: 0.8183
Validation Accuracy: 0.7561
Epoch 6/25, Loss: 0.7821
Validation Accuracy: 0.7951
Epoch 7/25, Loss: 0.7147
Validation Accuracy: 0.7756
Epoch 8/25, Loss: 0.6715
Validation Accuracy: 0.7854
Epoch 9/25, Loss: 0.6246
Validation Accuracy: 0.8390
Epoch 10/25, Loss: 0.6034
Validation Accuracy: 0.7951
Epoch 11/25, Loss: 0.5728
Validation Accuracy: 0.8488
Epoch 12/25, Loss: 0.5245
Validation Accuracy: 0.8244
Epoch 13/25, Loss: 0.4850
Validation Accuracy: 0.8439
Epoch 14/25, Loss: 0.4809
Validation Accuracy: 0.8878
Epoch 15/25, Loss: 0.4637
Validation Accuracy: 0.8878
Epoch 1

In [14]:
# === SETUP ===
import os
import cv2
import numpy as np
import mediapipe as mp
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score
from torch.utils.data import Dataset, DataLoader
import zipfile
import pickle

# === CONFIG ===
zip_path = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET.zip.zip"   # <-- your zip file path
extract_to = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET"     # <-- extraction folder
IMAGE_SIZE = (256, 256)
BATCH_SIZE = 32
EPOCHS = 25

# === EXTRACT ZIP ===
if not os.path.exists(extract_to):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(extract_to)
    print("Dataset extracted to:", extract_to)
else:
    print("Dataset already extracted")

# === AUTO FIND train/test ===
def find_subfolder_path(base_path, target):
    for root, dirs, files in os.walk(base_path):
        for d in dirs:
            if d.lower() == target.lower():
                return os.path.join(root, d)
    return None

TRAIN_PATH = find_subfolder_path(extract_to, "train")
TEST_PATH  = find_subfolder_path(extract_to, "test")

if not TRAIN_PATH or not TEST_PATH:
    raise FileNotFoundError("Could not find 'train' or 'test' folders")

print(f"TRAIN_PATH: {TRAIN_PATH}")
print(f"TEST_PATH : {TEST_PATH}")

# === MEDIAPIPE ===
mp_pose = mp.solutions.pose
pose_detector = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)

def extract_keypoints(image_path):
    image = cv2.imread(image_path)
    if image is None:
        print(f"Warning: Unable to read {image_path}")
        return None

    image = cv2.resize(image, IMAGE_SIZE)
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose_detector.process(image_rgb)

    if not results.pose_landmarks:
        return None

    keypoints = []
    for lm in results.pose_landmarks.landmark:
        keypoints.extend([lm.x, lm.y, lm.z, lm.visibility])

    return keypoints

def load_data(path):
    X, y = [], []
    labels = sorted(os.listdir(path))
    print(f"Found classes: {labels}")

    for label in labels:
        class_folder = os.path.join(path, label)

        if not os.path.isdir(class_folder):
            continue

        for file in os.listdir(class_folder):
            img_path = os.path.join(class_folder, file)
            keypoints = extract_keypoints(img_path)

            if keypoints is not None:
                X.append(keypoints)
                y.append(label)

    return np.array(X, dtype=np.float32), np.array(y), labels

# === LOAD DATA ===
print("Loading training data...")
X_train, y_train, class_names = load_data(TRAIN_PATH)

print("Loading test data...")
X_test, y_test, _ = load_data(TEST_PATH)

# === LABEL ENCODING ===
le = LabelEncoder()
y_train_enc = le.fit_transform(y_train)
y_test_enc  = le.transform(y_test)

num_classes = len(le.classes_)

with open("label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

# === DATASET ===
class PoseDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

train_loader = DataLoader(PoseDataset(X_train, y_train_enc), batch_size=BATCH_SIZE, shuffle=True)
test_loader  = DataLoader(PoseDataset(X_test, y_test_enc), batch_size=BATCH_SIZE)

# === MODEL ===
class PoseClassifier(nn.Module):
    def __init__(self, input_dim=132, hidden1=256, hidden2=128, output_dim=num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden1),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden1, hidden2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden2, output_dim)
        )

    def forward(self, x):
        return self.net(x)

# === DEVICE ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = PoseClassifier().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
loss_fn = nn.CrossEntropyLoss()

# === TRAINING ===
print("Training model...")

for epoch in range(EPOCHS):
    model.train()
    running_loss = 0.0

    for Xb, yb in train_loader:
        Xb, yb = Xb.to(device), yb.to(device)

        optimizer.zero_grad()
        preds = model(Xb)
        loss = loss_fn(preds, yb)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{EPOCHS}, Loss: {running_loss/len(train_loader):.4f}")

# === TEST ===
print("Evaluating on test set...")

model.eval()
y_preds, y_true = [], []

with torch.no_grad():
    for Xt, yt in test_loader:
        Xt, yt = Xt.to(device), yt.to(device)

        preds = model(Xt)

        y_preds.extend(torch.argmax(preds, 1).cpu().numpy())
        y_true.extend(yt.cpu().numpy())

test_acc = accuracy_score(y_true, y_preds)
print(f"Test Accuracy: {test_acc:.4f}")

# === SAVE ===
torch.save(model.state_dict(), "pose_classifier.pth")
print("Saved: pose_classifier.pth")

# === TORCHSCRIPT ===
example_input = torch.rand(1, 132).to(device)
scripted_model = torch.jit.trace(model, example_input)
scripted_model.save("pose_classifier_scripted.pt")

print("Saved: pose_classifier_scripted.pt")

Dataset already extracted
TRAIN_PATH: C:\Users\dell\Desktop\Yoga pose detection\DATASET\TRAIN
TEST_PATH : C:\Users\dell\Desktop\Yoga pose detection\DATASET\TEST
Loading training data...
Found classes: ['downdog', 'goddess', 'plank', 'tree', 'warrior2']
Loading test data...
Found classes: ['downdog', 'goddess', 'plank', 'tree', 'warrior2']
Training model...
Epoch 1/25, Loss: 1.4338
Epoch 2/25, Loss: 1.0217
Epoch 3/25, Loss: 0.8960
Epoch 4/25, Loss: 0.8048
Epoch 5/25, Loss: 0.7418
Epoch 6/25, Loss: 0.6871
Epoch 7/25, Loss: 0.6111
Epoch 8/25, Loss: 0.5808
Epoch 9/25, Loss: 0.5663
Epoch 10/25, Loss: 0.5295
Epoch 11/25, Loss: 0.4869
Epoch 12/25, Loss: 0.4680
Epoch 13/25, Loss: 0.4373
Epoch 14/25, Loss: 0.4234
Epoch 15/25, Loss: 0.3988
Epoch 16/25, Loss: 0.4143
Epoch 17/25, Loss: 0.3753
Epoch 18/25, Loss: 0.3811
Epoch 19/25, Loss: 0.3594
Epoch 20/25, Loss: 0.3475
Epoch 21/25, Loss: 0.3369
Epoch 22/25, Loss: 0.3511
Epoch 23/25, Loss: 0.3336
Epoch 24/25, Loss: 0.3336
Epoch 25/25, Loss: 0.3028


In [15]:
import torch
import cv2
import numpy as np

# Load the TorchScript model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = torch.jit.load("pose_classifier_scripted.pt").to(device)
model.eval()

# Load label encoder
import pickle
with open('label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

# MediaPipe pose setup (same as training)
mp_pose = mp.solutions.pose
pose_detector = mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5)

def extract_keypoints_from_image(image_path):
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Image not found or unable to read: {image_path}")
    image = cv2.resize(image, (256, 256))
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    results = pose_detector.process(image_rgb)
    if not results.pose_landmarks:
        raise ValueError("No pose landmarks detected.")
    keypoints = []
    for lm in results.pose_landmarks.landmark:
        keypoints.extend([lm.x, lm.y, lm.z, lm.visibility])
    return np.array(keypoints, dtype=np.float32)

def predict_pose(image_path):
    keypoints = extract_keypoints_from_image(image_path)
    input_tensor = torch.tensor(keypoints).unsqueeze(0).to(device)  # shape: [1, 132]
    with torch.no_grad():
        output = model(input_tensor)
        pred_class_idx = torch.argmax(output, dim=1).item()
    pred_class = le.inverse_transform([pred_class_idx])[0]
    return pred_class

# Example usage:
test_image_path = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET\TEST\goddess\00000000.jpg"  # Change path accordingly
try:
    prediction = predict_pose(test_image_path)
    print(f"Predicted class for image: {prediction}")
except Exception as e:
    print(f"Error during prediction: {e}")


Predicted class for image: goddess


In [16]:
import cv2
import mediapipe as mp
import numpy as np
import torch
import pickle
import os

# === MODEL CLASS (MUST MATCH TRAINING) ===
class PoseClassifier(torch.nn.Module):
    def __init__(self, input_dim=132, hidden1=256, hidden2=128, output_dim=10):  # ⚠️ change output_dim if needed
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden1),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(hidden1, hidden2),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(hidden2, output_dim)
        )

    def forward(self, x):
        return self.net(x)

# === DEVICE ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === LOAD LABEL ENCODER FIRST (to get correct num_classes) ===
with open("label_encoder.pkl", "rb") as f:
    le = pickle.load(f)

num_classes = len(le.classes_)

# === LOAD MODEL ===
model = PoseClassifier(output_dim=num_classes)
model.load_state_dict(torch.load("pose_classifier.pth", map_location=device))
model.to(device)
model.eval()

# === MEDIAPIPE ===
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

# === KEYPOINT EXTRACTION ===
def extract_keypoints(image_array):
    with mp_pose.Pose(static_image_mode=True, min_detection_confidence=0.5) as pose:
        image_rgb = cv2.cvtColor(image_array, cv2.COLOR_BGR2RGB)
        results = pose.process(image_rgb)

        if not results.pose_landmarks:
            return None, None

        keypoints = []
        for lm in results.pose_landmarks.landmark:
            keypoints.extend([lm.x, lm.y, lm.z, lm.visibility])

        return np.array(keypoints, dtype=np.float32).reshape(1, -1), results.pose_landmarks

# === DRAW LANDMARKS ===
def draw_pose_landmarks(image_array, landmarks):
    mp_drawing.draw_landmarks(
        image_array,
        landmarks,
        mp_pose.POSE_CONNECTIONS,
        mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=3),
        mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2)
    )

# === MAIN ===
if __name__ == "__main__":

    # 🔁 CHANGE THIS PATH
    image_path = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET\TEST\goddess\00000000.jpg"

    if not os.path.exists(image_path):
        print("❌ Image not found:", image_path)
        exit()

    img = cv2.imread(image_path)

    keypoints, landmarks = extract_keypoints(img)

    if keypoints is None:
        print("❌ No pose detected in image")

        cv2.imshow("Output", img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
        exit()

    # === PREDICTION ===
    input_tensor = torch.tensor(keypoints).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        pred_idx = output.argmax(dim=1).item()
        pred_label = le.inverse_transform([pred_idx])[0]

    # === DRAW + TEXT ===
    draw_pose_landmarks(img, landmarks)

    cv2.putText(
        img,
        f"Pose: {pred_label}",
        (30, 50),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.2,
        (0, 255, 0),
        3,
        cv2.LINE_AA
    )

    # === SAVE OUTPUT ===
    output_path = "output_pose.jpg"
    cv2.imwrite(output_path, img)

    print(f"✅ Detected Pose: {pred_label}")
    print(f"📁 Saved output: {output_path}")

    # === DISPLAY WINDOW ===
    cv2.imshow("Pose Detection", img)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

C:\Users\dell\AppData\Local\Temp\ipykernel_32964\3607473434.py:36: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("pose_classifier.pth", map_

✅ Detected Pose: goddess
📁 Saved output: output_pose.jpg


In [17]:
folder_path = r"C:\Users\dell\Desktop\Yoga pose detection\DATASET"

for filename in os.listdir(folder_path):
    path = os.path.join(folder_path, filename)

    img = cv2.imread(path)
    if img is None:
        continue

    keypoints, landmarks = extract_keypoints(img)

    if keypoints is None:
        print(f"{filename} → No pose detected")
        continue

    input_tensor = torch.tensor(keypoints).to(device)

    with torch.no_grad():
        output = model(input_tensor)
        pred_idx = output.argmax(dim=1).item()
        pred_label = le.inverse_transform([pred_idx])[0]

    draw_pose_landmarks(img, landmarks)

    cv2.putText(img, f"Pose: {pred_label}", (30, 50),
                cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0,255,0), 3)

    output_path = f"output_{filename}"
    cv2.imwrite(output_path, img)

    print(f"{filename} → {pred_label}")

In [18]:
import cv2
import mediapipe as mp
import numpy as np
import torch
import pickle

# === MODEL CLASS (same as training) ===
class PoseClassifier(torch.nn.Module):
    def __init__(self, input_dim=132, hidden1=256, hidden2=128, output_dim=10):
        super().__init__()
        self.net = torch.nn.Sequential(
            torch.nn.Linear(input_dim, hidden1),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(hidden1, hidden2),
            torch.nn.ReLU(),
            torch.nn.Dropout(0.3),
            torch.nn.Linear(hidden2, output_dim)
        )

    def forward(self, x):
        return self.net(x)

# === DEVICE ===
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === LOAD LABEL ENCODER ===
with open('label_encoder.pkl', 'rb') as f:
    le = pickle.load(f)

num_classes = len(le.classes_)

# === LOAD MODEL ===
model = PoseClassifier(output_dim=num_classes)
model.load_state_dict(torch.load("pose_classifier.pth", map_location=device))
model.to(device)
model.eval()

# === MEDIAPIPE SETUP ===
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

pose = mp_pose.Pose(
    static_image_mode=False,
    model_complexity=1,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# === START WEBCAM ===
cap = cv2.VideoCapture(0)

print("✅ Press 'q' to exit")

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Flip for mirror effect
    frame = cv2.flip(frame, 1)

    # Convert to RGB
    image_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

    # Process pose
    results = pose.process(image_rgb)

    if results.pose_landmarks:
        # === EXTRACT KEYPOINTS ===
        keypoints = []
        for lm in results.pose_landmarks.landmark:
            keypoints.extend([lm.x, lm.y, lm.z, lm.visibility])

        keypoints = np.array(keypoints, dtype=np.float32).reshape(1, -1)

        # === PREDICTION ===
        input_tensor = torch.tensor(keypoints).to(device)

        with torch.no_grad():
            output = model(input_tensor)
            pred_idx = output.argmax(dim=1).item()
            pred_label = le.inverse_transform([pred_idx])[0]

        # === DRAW LANDMARKS ===
        mp_drawing.draw_landmarks(
            frame,
            results.pose_landmarks,
            mp_pose.POSE_CONNECTIONS,
            mp_drawing.DrawingSpec(color=(0, 255, 0), thickness=2, circle_radius=2),
            mp_drawing.DrawingSpec(color=(0, 0, 255), thickness=2)
        )

        # === SHOW PREDICTION ===
        cv2.putText(
            frame,
            f"Pose: {pred_label}",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 255, 0),
            2,
            cv2.LINE_AA
        )

    else:
        cv2.putText(
            frame,
            "No Pose Detected",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            1.0,
            (0, 0, 255),
            2
        )

    # === DISPLAY ===
    cv2.imshow("AI Yoga Trainer", frame)

    # Press Q to exit
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

C:\Users\dell\AppData\Local\Temp\ipykernel_32964\4279323925.py:35: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load("pose_classifier.pth", map_

✅ Press 'q' to exit
